# Parte 5: Galeria de Falhas e Análise de Campo Receptivo

Neste notebook vamos diagnosticar por que a U-Net apresenta falsos positivos e quebras de instâncias em aglomerações gigantes. A teoria nos diz que redes puramente convolucionais possuem uma limitação fundamental baseada na forma como as convoluções e poolings se acumulam: o **Campo Receptivo (Receptive Field - RF)**.

## 1. Dedução Teórica do Campo Receptivo

Vamos usar as fórmulas vistas em aula para calcular o RF passo a passo.
*   $r_{l} = r_{l-1} + (k_l - 1) \cdot j_{l-1}$ (Tamanho do RF atual)
*   $j_l = j_{l-1} \cdot s_l$ (Distância/salto atual)

Começando com $r_0=1$ e $j_0=1$, traçamos a progressão através do Encoder da `UNet` (4 blocos de _DoubleConv_ + MaxPool):

| Camada | Kernel ($k$) | Stride ($s$) | Salto ($j$) | RF Teórico ($r$) |
| :--- | :---: | :---: | :---: | :---: |
| Conv1 | 3 | 1 | 1 | 3 |
| Conv2 | 3 | 1 | 1 | 5 |
| MaxPool1 | 2 | 2 | 2 | 6 |
| Conv3 | 3 | 1 | 2 | 10 |
| Conv4 | 3 | 1 | 2 | 14 |
| MaxPool2 | 2 | 2 | 4 | 16 |
| Conv5 | 3 | 1 | 4 | 24 |
| Conv6 | 3 | 1 | 4 | 32 |
| MaxPool3 | 2 | 2 | 8 | 36 |
| Conv7 | 3 | 1 | 8 | 52 |
| Conv8 | 3 | 1 | 8 | 68 |
| MaxPool4 | 2 | 2 | 16 | 76 |
| **Conv9 (Gargalo)** | 3 | 1 | 16 | 108 |
| **Conv10 (Gargalo)** | 3 | 1 | 16 | **140** |

**Conclusão:** O campo receptivo da nossa rede é estritamente limitado a um quadrado de **140x140 pixels**.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from skimage.measure import regionprops
import matplotlib.colors as mcolors

# Configuração de Paths
repo_root = Path(".").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pa1.data import make_dsb2018_loaders
from pa1.models import UNet
from pa1.utils import get_device
from pa1.postprocessing import watershed_to_instances
from pa1.config import load_config

## 2. Histograma de Diâmetros Reais vs. Campo Receptivo

Vamos calcular o diâmetro equivalente das instâncias no dataset de treino e comparar com o nosso limite teórico de 140px.

In [ ]:
cfg = load_config("pa1/config.yaml", parte="2")
_, _, test_loader = make_dsb2018_loaders(data_dir=cfg.data.data_dir, batch_size=1, num_workers=0)

diameters = []
for batch in test_loader:
    gt = batch["mask_instances"].squeeze().numpy()
    props = regionprops(gt)
    for p in props:
        diameters.append(getattr(p, "equivalent_diameter_area", p.equivalent_diameter))

plt.figure(figsize=(10, 5))
plt.hist(diameters, bins=50, color='skyblue', edgecolor='black')
plt.axvline(x=140, color='red', linestyle='--', linewidth=2, label='RF Limite Teórico (140px)')
plt.title("Histograma de Diâmetros dos Núcleos no DSB2018 vs RF Teórico")
plt.xlabel("Diâmetro Equivalente (px)")
plt.ylabel("Frequência")
plt.legend()
plt.show()

[DSB2018] Varrendo pa1/data/stage1_train ...


FileNotFoundError: Diretório de dados não encontrado: pa1/data/stage1_train
Verifique o caminho em config.yaml (chave data.data_dir).

**Análise:** Como o gráfico mostra, a imensa maioria dos núcleos individuais coube no nosso RF. Mas, o que acontece quando núcleos formam uma "massa" gigante aglomerada (ou quando um núcleo excepcionalmente grande aparece)? Vamos buscar as 5 piores imagens do teste da Trilha A.

In [ ]:
csv_path = Path("pa1/outputs/metrics/parte2_per_image_instance_metrics.csv")
if not csv_path.exists():
    raise FileNotFoundError("CSV não encontrado! Rode a avaliação da parte 2 antes.")

df = pd.read_csv(csv_path)
worst_5 = df.sort_values("mAP").head(5)
worst_indices = worst_5["idx"].tolist()
worst_5[["idx", "mAP", "count_error", "n_gt", "n_pred"]]

## 3. Galeria das Piores Falhas e o Problema em Ação

Carregaremos o modelo congelado (Trilha A padrão) e vamos extrair os mapas previstos nessas imagens críticas.

In [ ]:
device = get_device()
model = UNet(in_channels=3, out_channels=3, dilate_bottleneck=1).to(device)
ckpt_path = Path("pa1/outputs/checkpoints/parte2_baseline_unet.pt")
model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
model.eval()

np.random.seed(42)
colors = np.random.rand(10000, 3)
colors[0] = [0, 0, 0] 
cmap = mcolors.ListedColormap(colors)

for idx in worst_indices:
    item = test_loader.dataset[idx]
    img_tensor = item["image"].unsqueeze(0).to(device)
    gt_inst = item["mask_instances"].numpy()
    
    with torch.no_grad():
        logits = model(img_tensor)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
        
    pred_inst = watershed_to_instances(probs, interior_channel=1, marker_threshold=0.6, min_area=15)
    
    # Plot
    fig, axs = plt.subplots(1, 4, figsize=(16, 4))
    rgb_img = np.clip(item["image"].permute(1, 2, 0).numpy(), 0, 1)
    
    axs[0].imshow(rgb_img)
    axs[0].set_title(f"Imagem IDX: {idx}")
    axs[0].axis("off")
    
    axs[1].imshow(gt_inst, cmap=cmap, interpolation="nearest")
    axs[1].set_title("Ground Truth")
    axs[1].axis("off")
    
    axs[2].imshow(probs[2], cmap="magma")
    axs[2].set_title("Probabilidade Fronteira")
    axs[2].axis("off")
    
    axs[3].imshow(pred_inst, cmap=cmap, interpolation="nearest")
    axs[3].set_title("Watershed Previsto")
    axs[3].axis("off")
    
    plt.tight_layout()
    plt.show()

## 4. Intervenção: Convoluções Dilatadas no Gargalo

Baseado no diagnóstico, percebemos que o RF de 140px nem sempre fornece contexto global suficiente para que o modelo entenda grandes aglomerados. Em vez de adicionar camadas novas, usamos *Atrous (Dilated) Convolutions* no bottleneck (`dilation=4`).

Matematicamente, alterar o último bloco de convoluções para `dilation=4` faz o kernel $3\times 3$ atuar como $9\times 9$ com buracos.
- Conv9 (Dilated $d=4$): O $k_{	ext{eff}} = 1 + (3-1)\times 4 = 9$. O novo RF será $r_{13} = 76 + (9-1)\times 16 = 204$.
- Conv10 (Dilated $d=4$): O novo RF final será $r_{14} = 204 + 8\times 16 = \mathbf{332}$.

Dobramos o campo receptivo da nossa rede gastando 0 parâmetros a mais. Vamos instanciar esse novo modelo e testar se ele prevê fronteiras com maior coerência topológica nas falhas.

In [ ]:
# Instancia novo modelo com Dilation
model_dilated = UNet(in_channels=3, out_channels=3, dilate_bottleneck=4).to(device)

print("Nova arquitetura de bottleneck com dilation=4:")
print(model_dilated.bottleneck)

print("\n[Aviso] Para avaliação quantitativa completa desta intervenção, seria necessário treinar esse modelo. Na próxima célula faremos uma avaliação pass-through apenas para confirmar a mecânica de propagação.")